##<img src="https://img.stackshare.io/service/10345/ADB.png"/></a> Workshop Lab 2 - Pipeline Interactive Demo

## Data processing framework - Lakehouse Declarative Pipeline

* Modern data architecture
* Key termonologies and approchaes
* MiniUsecase - Medallion architecture with LDP
    * Architecture Plan and prerequisties check
    * LDP GUI check
    * Create a declarative pipeline via the layers
    * Monitoring and Error handling - GUI and Logs
    * Data Quality - Expectations (Guards)
    * Data Exploration via SQL Editor

<h2 style="text-align: center;">Lakehouse - Medallion architecture (multi-hop approach)</h2> 

<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://databricks.com/wp-content/uploads/2019/08/Delta-Lake-Multi-Hop-Architecture-Bronze.png" alt="LakeHouse" style="width: 600px height: 600px">
</div>



##Lakehouse supports
#
* Transaction support
* Schema enforcement and governance
* BI support
* Storage is decoupled from compute
* Openness
* Support for diverse data types ranging from unstructured to structured data
* Support for diverse workloads
* End-to-end streaming


# 🧭 ETL in Databricks

---

### **Core Concepts**

**Orchestration**  
The coordination and scheduling of multiple data processing tasks or workflows to ensure they run in the correct order and timing.

**Job**  
A single, executable unit of work in Databricks (e.g., running a notebook, script, or pipeline) that can be scheduled or triggered.

**Pipeline**  
A defined sequence of data transformations and movement steps (e.g., ETL/ELT) that process data from source to target, often managed and executed through orchestration.

---

### **Declarative Frameworks**

**LDP Lakeflow (Spark) Declarative Pipelines DLT ←→ Delta Live Tables  **

**Delta Live Tables (DLT) - kind of legacy**  
*A declarative framework for building reliable, maintainable, and testable data processing pipelines.*

**Lakeflow Declarative Pipelines (LDP)**  
*A declarative framework for developing and running batch and streaming data pipelines in SQL and Python. Built on the Apache Spark Declarative Pipeline*

---

### **Approach Styles**

**Procedural approach**  
You define *how* tasks should run by specifying the exact sequence of operations and control flow.

**Declarative approach**  
You describe *what* result you want, and the system figures out how to execute it optimally.

---

### **Pipeline objects**

**Sink**  
A target destination (e.g., Delta table, Kafka topic, Event Hub) for streaming flows where processed records are written out via a pipeline.

**Streaming table**  
A Unity Catalog–managed table defined for incremental or streaming ingestion and processing, backed by a serverless pipeline, and refreshed on a schedule or trigger.  

**Materialized view**  
A persistent table that stores the precomputed results of a query, updates incrementally or on schedule, and provides faster query performance than a standard view.

**Flow**  
A named data path that reads from a source, applies processing logic, and writes results into a dataset. Every streaming table and materialized view has one default flow (named after its target); extra **append flows** can write into the same streaming table (fan-in, new sources, backfills).


<div align="center">
  <img src="https://docs.databricks.com/aws/en/assets/images/dlt-core-concepts-6bc9894d3682035cadac19f3980875ae.png" width="800"/>
</div>

<div align="center">
<img src="https://miro.medium.com/v2/resize:fit:4800/format:webp/1*MggtENQXxtieCKA30b4xbg.png" width="800"/>
</div>

#Mini Usecase
![image_1777449282252.png](../images/image_1777449282252.png "image_1777449282252.png")
## Steps:
  * Emulate the data source with Notebook cell
  * Create medallion layers via Lakeflow Pipelines (old terminology: DLT) (bronze, silver, gold):
  * Run it and test


In [0]:
# Idempotent: safe to run anytime (re-creates the volume and folder if missing)
# Create the source layer volume and folder
catalog = "workspace"
schema = "default"
volume_folder = f"/Volumes/{catalog}/{schema}/raw_data"

spark.sql(f"CREATE VOLUME IF NOT EXISTS `{catalog}`.`{schema}`.`raw_data`")

def ensure_empty_folder(path):
    # Remove any existing contents (no-op if the folder doesn't exist yet)
    dbutils.fs.rm(path, True)
    # Create the folder and any missing parents
    dbutils.fs.mkdirs(path)


ensure_empty_folder(f"/Volumes/{catalog}/{schema}/raw_data/customers")


In [0]:
#Check
dbutils.fs.ls(f"/Volumes/{catalog}/{schema}/raw_data/customers")

In [0]:
#Step 0 - Generate some source data
%pip install faker
from pyspark.sql import functions as F
from faker import Faker
from collections import OrderedDict
import uuid
import random

#Configure fields
fake = Faker()
# docs: https://github.com/xfxf/faker-python

fake_firstname = F.udf(lambda: fake.first_name())
fake_lastname = F.udf(lambda: fake.last_name())
fake_email = F.udf(lambda: fake.ascii_company_email())
#fake_birth_date = F.udf(lambda:fake.date_time_this_century().strftime("%m-%d-%Y"))
fake_birth_date = F.udf(lambda:fake.date_between_dates(date_start="-90y", date_end="-2y").strftime("%m-%d-%Y"))
# Limited country list for cleaner grouping/analysis
available_eu_countries = [
    "Hungary", "Germany", "Austria", "Slovakia", "Poland",
    "Czech Republic", "Romania", "France", "Italy", "Spain",
    "Netherlands", "Slovenia", "Belgium", "Greece", "Portugal", "Sweden", "Norway", "Denmark", "Finland", "Ireland", "United Kingdom", "Luxembourg", "Malta", "Iceland", "Liechtenstein", "Andorra", "Monaco", "San Marino", "Vatican City", "Bulgaria", "Cyprus", "Estonia", "Latvia", "Lithuania", "Albania", "Bosnia and Herzegovina"
]
fake_country = F.udf(lambda: random.choice(available_eu_countries))
fake_registration_date = F.udf(lambda:fake.date_time_this_decade().strftime("%m-%d-%Y %H:%M:%S"))
fake_id = F.udf(lambda: str(uuid.uuid4()) if random.uniform(0, 1) < 0.98 else None)
categories = OrderedDict([("RED", 0.5),("GREEN", 0.1),("BLUE", 0.3),(None, 0.01)])
fake_categories = F.udf(lambda:fake.random_elements(elements=categories, length=1)[0])

# Set the record number
number_of_customer = 99
#optional performance step
df = spark.range(0, number_of_customer).repartition(2)

df = df.withColumn("id", fake_id())
df = df.withColumn("firstname", fake_firstname())
df = df.withColumn("lastname", fake_lastname())
df = df.withColumn("date_of_birth", fake_birth_date())
df = df.withColumn("email", fake_email())
df = df.withColumn("country_of_origin", fake_country())
df = df.withColumn("customer_category", fake_categories())
df = df.withColumn("registration_date", fake_registration_date())
df_customers = df.withColumn("load_ts", F.current_timestamp())

print(df_customers.count())
# You can play with the number of customers and partitions to generate to see how the performance scales
df_customers.repartition(5).write.format("json").mode("append").save(volume_folder+"/customers")

print(f"{number_of_customer} customers are generated. Process done..")

In [0]:
display(dbutils.fs.ls(volume_folder+"/customers"))

In [0]:
# Challenge
# Check the data

df = spark.read.format("json").load(volume_folder+"/customers")
#df.groupby("country_of_origin").count().orderBy("count").display()
df.limit(10).display()

In [0]:
# Clean up
# Remove the generated data
dbutils.fs.rm(volume_folder + "/customers", True)

# Drop the volume (also deletes all files stored in it)
spark.sql(f"DROP VOLUME IF EXISTS `{catalog}`.`{schema}`.`raw_data`")

spark.sql("DROP TABLE IF EXISTS workspace.default.bronze_customers")
spark.sql("DROP TABLE IF EXISTS workspace.default.silver_customers")
spark.sql("DROP TABLE IF EXISTS workspace.default.customers_categories") 
spark.sql("DROP TABLE IF EXISTS workspace.default.customers_registration_monthly") 
spark.sql("DROP TABLE IF EXISTS workspace.default.customers_by_country") 




### Source data: `customers` (JSON)

Each run of **Step 0** appends `number_of_customer` (default 1000) fake customer records as JSON files under `{volume_folder}/customers`. Some data-quality quirks are added on purpose: ~2% of rows have a `null` id, some rows have a `null` `customer_category`, and dates are stored as strings rather than typed values.

| Field | Description |
| --- | --- |
| `id` | Unique customer identifier (UUID string). ~2% of rows are `null` to simulate records with a missing key. |
| `firstname` | Fake first name (Faker). |
| `lastname` | Fake last name (Faker). |
| `date_of_birth` | Fake date within the current century, stored as string in `MM-DD-YYYY` format (not a date type). |
| `email` | Fake company email address (Faker `ascii_company_email`). |
| `country_of_origin` | One of 36 fixed European countries (random pick from a limited list), so country grouping stays clean. |
| `customer_category` | Weighted random category: `RED` (~55%), `BLUE` (~33%), `GREEN` (~11%), `null` (~1%) — tests null handling and filtering downstream. |
| `registration_date` | Fake datetime within the current year, stored as string in `MM-DD-YYYY HH:MM:SS` format (not a timestamp type). |
| `load_ts` | Timestamp of record generation (`current_timestamp()`), typed as timestamp — usable as the batch/ingestion marker. |

#Q&A


contacts: tamas.papp@t-systems.com, 
         agnes.gerse@t-systems.com

# Data Quality basics

The four classic dimensions describe *what* can be wrong with data. Expectations in Lakeflow pipelines are the *mechanism* to check and enforce them. Our source data contains intentional quirks — one for each dimension:

**Completeness** — required values are present; nothing critical is missing.  
*In our source:* ~2% of `id`s and ~1% of `customer_category` values are `null` on purpose.  
*Enforced in silver:* `valid_id: id IS NOT NULL` and `valid_category: customer_category IN ('RED','GREEN','BLUE')` — both drop the offending rows.

**Uniqueness** — no unintended duplicates (keys, identifiers).  
*In our source:* `id` is a fresh UUID per record, but `email` can legitimately repeat across customers.  
*Caveat:* expectations are evaluated **row by row**, so uniqueness (a dataset-level property) can't be checked with a simple expectation — use an aggregate check MV (e.g. `COUNT(*) vs COUNT(DISTINCT id)`) or deduplicate upstream.

**Validity** — values conform to the expected domain, format or range.  
*In our source:* `customer_category` outside `{RED, GREEN, BLUE}`, `email` without `@`, unparseable dates.  
*Enforced in silver:* `valid_category` covers the category domain; format checks could be added, e.g. `email LIKE '%@%'`.

**Consistency** — values agree within a record and across tables/loads (same formats, same encodings, no contradictions).  
*In our source:* dates are plain strings (`MM-DD-YYYY`, `MM-DD-YYYY HH:MM:SS`) instead of typed dates — risky for joins and comparisons — while `load_ts` is properly typed.  
*Handled downstream:* silver keeps a stable schema across loads; gold MVs derive from silver, so their numbers always reconcile with it.

| Dimension | Question it answers | Example in `customers` | Where it's handled |
| --- | --- | --- | --- |
| Completeness | Is anything missing? | `null` `id` (~2%), `null` category (~1%) | row-level expectation → drop |
| Uniqueness | Is anything duplicated? | repeated `email`s | aggregate check MV / dedup (not row-level) |
| Validity | Are values in the allowed domain? | category not in {RED, GREEN, BLUE} | row-level expectation → drop |
| Consistency | Do formats and agreements hold? | dates as strings vs typed `load_ts` | schema alignment + typed parsing |

# Available expectations in Lakeflow pipelines

Six operators on two axes: **action on violation** × **single vs. multiple rules**.

| Operator | Action on violation | Rules |
| --- | --- | --- |
| `@dp.expect(desc, expr)` | **WARN** — row is kept, violation counted & logged | single |
| `@dp.expect_or_drop(desc, expr)` | **DROP** — row removed before writing | single |
| `@dp.expect_or_fail(desc, expr)` | **FAIL** — update stops, transaction rolls back | single |
| `@dp.expect_all({desc: expr, ...})` | WARN for each rule separately | multiple |
| `@dp.expect_all_or_drop({desc: expr, ...})` | DROP if *any* rule fails | multiple |
| `@dp.expect_all_or_fail({desc: expr, ...})` | FAIL if *any* rule fails | multiple |

Three ways to attach them:

1. **Decorator on the dataset function** (materialized view / table):
```python
@dp.materialized_view(name="customers_clean")
@dp.expect_or_drop("valid_email", "email LIKE '%@%'")
def customers_clean(): ...
```

2. **Keyword on `create_streaming_table`** (our silver pattern):
```python
dp.create_streaming_table(
  name="silver_customers",
  expect_all_or_drop={"valid_id": "id IS NOT NULL",
                      "valid_category": "customer_category IN ('RED', 'GREEN', 'BLUE')"})
```

3. **SQL constraint clause** (DBSQL or pipeline SQL files):
```sql
CREATE OR REFRESH STREAMING TABLE silver (
  CONSTRAINT valid_id EXPECT (id IS NOT NULL) ON VIOLATION DROP ROW
) AS SELECT ... ;
```

**Notes:**

* A constraint is a SQL boolean expression evaluated **per row** — no subqueries, no UDFs.
* Each expectation reports valid/dropped/failed record counts in the pipeline UI for every update, and in the event log.
* `expect_or_fail` stops only its own flow — other flows in the pipeline still commit.
* Expectation decorators only wrap dataset-defining functions. On a streaming table built with `create_streaming_table` + append flows (like our `silver_customers`), attach them via the keyword form above.


# Handling bad rows: drop or quarantine?

Expectations only **count** bad rows — they never write them anywhere. What happens to the rejected data is a design decision:

| Strategy | What happens to bad rows | Use when |
| --- | --- | --- |
| **Drop** (`expect_or_drop`) | removed before writing; only valid/dropped counts remain in metrics & event log | bad rows are noise (test data, junk records) — you never need them again |
| **Warn** (`expect`) | kept in the target, violations logged/monitored | soft quality signals, monitoring trends |
| **Fail** (`expect_or_fail`) | update stops, transaction rolls back | a violation means a serious upstream bug — nothing downstream should run |
| **Quarantine** | bad rows land in a separate table for review/reprocessing | compliance, finance, late corrections — you must be able to inspect and fix rejected records |

**Quarantine pattern:** expectations can't route rows, so you define a second streaming table whose flow reads the *same* source but keeps only the bad rows. Each flow maintains its own checkpoint, so silver (good rows) and quarantine (bad rows) both consume the source independently:

```python
# silver_quarantine.py — rejected records for review & reprocessing
dp.create_streaming_table(
  name="silver_customers_quarantine",
  comment="Rows failing the silver data quality rules")

@dp.append_flow(
  target="silver_customers_quarantine",
  name="silver_quarantine_flow"
)
def silver_quarantine_flow():
  return (
    spark.readStream.table("bronze_customers")
      .select("id", "firstname", "lastname", "country_of_origin",
              "customer_category", "email", "registration_date", "load_ts")
      .where("NOT (id IS NOT NULL AND customer_category IN ('RED', 'GREEN', 'BLUE'))")
  )
```

The clean flow (`silver_customers`) keeps its expectations and drops bad rows; the quarantine flow captures exactly those rows — the same rules, just negated in the `WHERE` clause.

**Decision guide:** noise → drop · must inspect/fix later → quarantine (+ optionally drop) · violation = upstream bug → fail. In our silver the ~2% null-id and invalid-category rows are currently dropped and lost — quarantine would keep them inspectable.

Docs:
- https://docs.databricks.com/aws/en/getting-started/data-pipeline-get-started
- https://youtu.be/krzr52wd4nM
- https://docs.databricks.com/aws/en/data-engineering/#gsc.tab=
- https://docs.databricks.com/aws/en/assets/images/dlt-core-concepts-6bc9894d3682035cadac19f3980875ae.png
- https://docs.databricks.com/aws/en/ldp/concepts#gsc.tab=0
- https://docs.databricks.com/aws/en/ldp/develop#gsc.tab=0